# Noise-aware, uncertainty-calibrated tangent-geometry selector

**Purpose:** Fix the *cross-noise-scale selection* problem on a **known-geometry synthetic manifold**, without retraining the score network.

This notebook reads `all_per_point_results.csv` previously saved by the *Ground-Truth Tangent Recovery* experiment. It uses **true tangent errors only for fitting/calibration on synthetic points**, and keeps 100 other points untouched for evaluation. The predictor uses no true error at inference.

**Important:** This establishes same-manifold synthetic performance only, not accurate MNIST dimension or cross-manifold generalization.

**Splits**: among 40 original fit points, 20 train and 20 calibration; original 100 test points stay independent. **Never split rows randomly**, since each point appears at three noise scales.

In [ ]:
from pathlib import Path
import importlib.util

try:
    from google.colab import drive, files
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive')
except ImportError:
    ROOT = Path.cwd()

# Original Ground-Truth Tangent Recovery notebook saved this CSV to Drive.
DATA = ROOT / 'synthetic_tangent_ground_truth' / 'all_per_point_results.csv'
OUT = ROOT / 'noise_aware_tangent_selection'
OUT.mkdir(parents=True, exist_ok=True)

if not DATA.exists():
    print('Expected CSV not found:', DATA)
    if importlib.util.find_spec('google.colab'):
        print('Upload all_per_point_results.csv from your earlier notebook.')
        uploaded = files.upload()
        DATA = Path('/content') / next(k for k in uploaded if k.endswith('.csv'))
    else:
        raise FileNotFoundError('Edit DATA to point to your all_per_point_results.csv')

print('Data:', DATA)
print('Output:', OUT)

## 1. Predictor and calibration algorithm

1. **Train only:** compare a Ridge and Random Forest regression using group-wise cross-validation. Predict true geometric error from noise scale, chord proxy, projector defect, and soft dimension. Select the predictor using **training points only**.
2. **Calibrate only:** calculate a point-level **simultaneous conformal residual** across all three noise scales. This creates a marginal, same-population upper prediction bound.
3. **Test only:** select the scale minimizing the predicted upper bound; abstain if it exceeds a predeclared tangent-error tolerance. Report true projector error, coverage, and comparison against raw proxy and projector-defect selectors.

The chosen tolerance `0.15` is an illustrative strict geometric target (about 6.1 degrees for rank-one tangent projectors); it is **not** derived from MNIST labels.

In [ ]:
%%writefile /content/Noise_Aware_Tangent_Selector.py
"""Geometry scale selection with held-out true tangent errors.

Run:
python Noise_Aware_Tangent_Selector.py --csv all_per_point_results.csv --out results

No score-model training or inference. Uses saved synthetic observations only.
Statistical coverage is marginal for an exchangeable point from the same synthetic
setting (and with all three sigmas observed), not distribution-free on MNIST.
"""
import argparse, json, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

FEATS = ['sigma','neighbor_proxy','projector_defect','soft_dimension']
LABEL = 'tangent_error'
REQUIRED = {'model','split','index','sigma',LABEL,'neighbor_proxy','projector_defect','soft_dimension'}


def load_data(path):
    df = pd.read_csv(path)
    missing = REQUIRED-set(df.columns)
    if missing: raise ValueError(f'Missing columns: {sorted(missing)}')
    df = df[df['model']=='learned_score'].copy()
    if df.empty: raise ValueError('No rows with model=learned_score')
    if df[list(REQUIRED - {'model','split'})].isna().any().any():
        raise ValueError('Found missing numbers in required columns')
    if df.duplicated(['split','index','sigma']).any():
        raise ValueError('Duplicate point/sigma rows')
    for sp in ['fit','test']:
        chunk=df[df.split==sp]
        if chunk.empty: raise ValueError(f'Missing {sp} split')
        counts=chunk.groupby('index')['sigma'].nunique()
        if counts.nunique()!=1 or counts.iloc[0]!=df.sigma.nunique():
            raise ValueError(f'{sp} must contain the same complete sigma grid per point')
    return df


def create_splits(df,seed=73):
    fit_ids = sorted(df.loc[df.split=='fit','index'].unique())
    # deterministic image-wise split, not row-wise split
    rng = np.random.default_rng(seed+159)
    ordered = rng.permutation(fit_ids)
    ntrain = max(10, int(np.floor(0.5*len(ordered))))
    if len(ordered)-ntrain<9:
        raise ValueError('At least 19 distinct fitting points needed to allow train+conformal splits')
    train_ids=set(ordered[:ntrain]); cal_ids=set(ordered[ntrain:])
    train=df[(df.split=='fit')&df['index'].isin(train_ids)].copy()
    calib=df[(df.split=='fit')&df['index'].isin(cal_ids)].copy()
    test=df[df.split=='test'].copy()
    return train,calib,test


def choose_predictor(train,seed=73):
    candidates = {
        'ridge': make_pipeline(StandardScaler(),Ridge(alpha=3.0)),
        'random_forest': RandomForestRegressor(n_estimators=120,max_depth=5,
                   min_samples_leaf=3,random_state=seed,n_jobs=-1),
    }
    folds=min(5,train['index'].nunique())
    cv=GroupKFold(n_splits=folds)
    res={}
    for name,base in candidates.items():
        losses=[]
        for tr,va in cv.split(train[FEATS],train[LABEL],groups=train['index']):
            m=clone(base);m.fit(train.iloc[tr][FEATS],train.iloc[tr][LABEL]);
            pred=m.predict(train.iloc[va][FEATS]);
            losses.append(np.mean(np.abs(train.iloc[va][LABEL].to_numpy()-pred)))
        res[name]=float(np.mean(losses))
    selected=min(res,key=res.get)
    model=clone(candidates[selected]);model.fit(train[FEATS],train[LABEL])
    return model,selected,res


def add_pred(df,model):
    v=df.copy()
    v['predicted_error']=np.clip(model.predict(v[FEATS]),0,math.sqrt(2))
    return v


def calibrate(train, calib, alpha=.10):
    # sigma-dependent scale comes only from TRAINING residuals.
    per_sigma = train.assign(residual=lambda d:d[LABEL]-d['predicted_error']).groupby('sigma')\
        ['residual'].agg(lambda a:np.sqrt(np.mean(np.square(a))))
    floors={float(s):max(.01,float(v)) for s,v in per_sigma.items()}
    calib=calib.copy()
    calib['uncertainty_scale']=calib['sigma'].map(floors)
    calib['scaled_abs_residual']=np.abs(calib[LABEL]-calib['predicted_error'])/calib['uncertainty_scale']
    # A maximum over scales protects selection of ANY sigma after calibration.
    scores=calib.groupby('index')['scaled_abs_residual'].max().sort_values().to_numpy()
    n=len(scores)
    k=math.ceil((n+1)*(1-alpha))
    q=math.inf if k>n else float(scores[k-1])
    return floors,q,n,calib


def evaluate_selector(test,floors,q,tau):
    test=test.copy()
    test['uncertainty_scale']=test['sigma'].map(floors)
    test['predicted_upper_bound'] = test['predicted_error']+q*test['uncertainty_scale']
    test['simultaneous_bound_covered']=(test[LABEL]<=test['predicted_upper_bound'])
    methods={
        'raw_neighbor_proxy':'neighbor_proxy',
        'raw_projector_defect':'projector_defect',
        'noise_aware_predicted_error':'predicted_error',
        'calibrated_upper_bound':'predicted_upper_bound',
    }
    cols=['index','sigma',LABEL,'predicted_error','predicted_upper_bound',
          'neighbor_proxy','projector_defect','soft_dimension']
    pieces=[]
    for name,key in methods.items():
        z=test.loc[test.groupby('index')[key].idxmin(),cols].copy()
        z['method']=name
        z['accepted']=(z.predicted_upper_bound<=tau) if name=='calibrated_upper_bound' else True
        pieces.append(z)
    oracle=test.loc[test.groupby('index')[LABEL].idxmin(),cols].copy()
    oracle['method']='oracle_true_error_UNAVAILABLE_ON_MNIST';oracle['accepted']=True
    pieces.append(oracle)
    selected=pd.concat(pieces,ignore_index=True)
    summary=[]
    for name,z in selected.groupby('method'):
        acc=z[z.accepted]
        summary.append({
           'method':name,'coverage':float(len(acc)/len(z)),
           'accepted_count':int(len(acc)),
           'mean_true_error_accepted':float(acc[LABEL].mean()) if len(acc) else np.nan,
           'mean_true_error_all_selected':float(z[LABEL].mean()),
           'fraction_accepted_above_tau':float(np.mean(acc[LABEL]>tau)) if len(acc) else np.nan,
           'sigma_counts_accepted':json.dumps({str(k):int(v) for k,v in acc.sigma.value_counts().sort_index().items()}),
        })
    coverage_by_point=test.groupby('index')['simultaneous_bound_covered'].all()
    return selected,pd.DataFrame(summary).sort_values('mean_true_error_all_selected'),float(coverage_by_point.mean())


def pair_bootstrap(selected,alpha=.05,nboot=4000,seed=73):
    grid=selected.pivot(index='index',columns='method',values=LABEL)
    d=grid['raw_neighbor_proxy']-grid['calibrated_upper_bound']
    vals=d.to_numpy()
    rng=np.random.default_rng(seed+716)
    draws=rng.integers(0,len(vals),size=(nboot,len(vals)))
    boot=vals[draws].mean(axis=1)
    return float(vals.mean()), np.quantile(boot,[alpha/2,1-alpha/2]).tolist()


def run(csv,out,alpha=.10,tau=.15,seed=73):
    out=Path(out);out.mkdir(parents=True,exist_ok=True)
    df=load_data(csv)
    tr,ca,te=create_splits(df,seed)
    model,name,cv=choose_predictor(tr,seed)
    tr=add_pred(tr,model)
    ca=add_pred(ca,model)
    te=add_pred(te,model)
    scales,q,n,ca=calibrate(tr,ca,alpha)
    selected,summary,observed= evaluate_selector(te,scales,q,tau)
    diff,ci=pair_bootstrap(selected,seed=seed)
    tr.to_csv(out/'train_points.csv',index=False)
    ca.to_csv(out/'calibration_points.csv',index=False)
    te.to_csv(out/'test_points_with_predictions.csv',index=False)
    selected.to_csv(out/'test_selected_by_method.csv',index=False)
    summary.to_csv(out/'method_comparison.csv',index=False)
    report={'source_csv':str(csv),'manifold':'circle R8, learned_score',
      'n_train_points':int(tr['index'].nunique()),'n_calib_points':int(ca['index'].nunique()),
      'n_test_points':int(te['index'].nunique()),'predictor':name,
      'train_only_cross_validation_MAE':cv,'sigmas':sorted(df.sigma.unique().tolist()),
      'alpha':alpha,'requested_error_tolerance':tau,'conformal_q':q,
      'sigma_residual_scales':{str(k):v for k,v in scales.items()},
      'observed_test_simultaneous_coverage':observed,
      'mean_paired_true_error_improvement_vs_raw_proxy':diff,
      'paired_bootstrap_CI95':ci,
      'limitation':'Coverage applies marginally to same-population points, NOT conditional on acceptance, NOT MNIST. The model uses true synthetic tangent labels in fitting and calibration.'}
    (out/'report.json').write_text(json.dumps(report,indent=2))
    print('Selection results on untouched test points:')
    print(summary.to_string(index=False))
    print('\n',json.dumps(report,indent=2))
    fig,axes=plt.subplots(1,2,figsize=(12,4.2))
    plot=summary[~summary['method'].str.contains('oracle')].sort_values('mean_true_error_all_selected')
    axes[0].barh(plot.method,plot.mean_true_error_all_selected)
    axes[0].set(xlabel='Mean TRUE tangent-projector error (all selected)',title='Held-out selection quality')
    axes[0].invert_yaxis()
    st=te.groupby('sigma').agg(true_error=(LABEL,'mean'),predicted_error=('predicted_error','mean'))
    axes[1].plot(st.index,st.true_error,marker='o',label='Actual error')
    axes[1].plot(st.index,st.predicted_error,marker='o',label='Predicted error')
    axes[1].set(xlabel='Noise sigma',ylabel='Mean tangent error',title='Scale-aware prediction (test)')
    axes[1].legend();axes[1].grid(alpha=.2)
    fig.tight_layout();fig.savefig(out/'selector_validation.png',dpi=170,bbox_inches='tight');plt.close(fig)
    print('Saved all reports to',out)
    return summary,report

if __name__=='__main__':
    p=argparse.ArgumentParser()
    p.add_argument('--csv',required=True)
    p.add_argument('--out',default='noise_aware_selector_results')
    p.add_argument('--alpha',type=float,default=.1)
    p.add_argument('--tau',type=float,default=.15)
    p.add_argument('--seed',type=int,default=73)
    a=p.parse_args();run(a.csv,a.out,a.alpha,a.tau,a.seed)


In [ ]:
import subprocess, sys
subprocess.run([
    sys.executable, '/content/Noise_Aware_Tangent_Selector.py',
    '--csv', str(DATA), '--out', str(OUT),
    '--alpha','0.10', '--tau','0.15', '--seed','73'
], check=True)

In [ ]:
import json, pandas as pd
from IPython.display import display, Image
report=json.loads((OUT/'report.json').read_text())
summary=pd.read_csv(OUT/'method_comparison.csv')
display(summary)
print('Predictor selected on TRAIN ONLY:',report['predictor'])
print('Conformal multiplier:',report['conformal_q'])
print('Observed held-out simultaneous coverage:',report['observed_test_simultaneous_coverage'])
print('Paired true-error improvement vs raw proxy:',report['mean_paired_true_error_improvement_vs_raw_proxy'])
print('95% bootstrap CI:',report['paired_bootstrap_CI95'])
display(Image(filename=str(OUT/'selector_validation.png')))

## 2. Interpretation rules

- **Improvement:** the corrected selector has lower **actual** held-out tangent-projector error than the raw-proxy selector, preferably with a positive paired bootstrap confidence interval.
- **Confidence:** the 90% simultaneous conformal bound is **marginal** for a new point from the **same** circle/noise/model distribution. It does **not** certify 90% conditional accuracy among accepted samples, and does not transfer automatically to MNIST.
- **If the bootstrap interval includes zero:** do **not** claim a statistically confirmed improvement.
- **Even if successful:** this is *stage 1*. Next train/test across **different manifolds** (ellipses, spheres, curved surfaces), and freeze the predictor before applying it to MNIST.
- **No MNIST classifier or surrogate is used** anywhere in this geometry experiment.